# SupplyMind AI — Preprocessing

Demonstrate the transformations applied immediately before model fitting:
chronological splitting, stateful feature engineering, one-hot encoding,
numerical imputation/scaling, and protection against unseen categories.

In [1]:
# -------------------
# Imports
# -------------------

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from supplymind.features.predictions.domain.constants import (
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
    TARGET_COLUMN,
)
from supplymind.features.predictions.ml.workflow import load_clean_syndelay

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [2]:
# -------------------
# Project configuration
# -------------------

DATASET_PATH = Path("../data/raw/syndelay/syndelay_v1.csv")
REPORT_ROOT = Path("../reports")

assert DATASET_PATH.exists(), (
    f"Dataset not found: {DATASET_PATH}"
)

In [3]:
from supplymind.features.predictions.ml.features import ShipmentFeatureEngineer
from supplymind.features.predictions.ml.preprocessing import build_preprocessor
from supplymind.features.predictions.ml.workflow import (
    load_clean_syndelay,
    prepare_model_data,
)

df = load_clean_syndelay(DATASET_PATH)
data = prepare_model_data(df)

## 1. Chronological split

In [4]:
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [
        len(data.X_train),
        len(data.X_validation),
        len(data.X_test),
    ],
    "positive_rate": [
        data.y_train.mean(),
        data.y_validation.mean(),
        data.y_test.mean(),
    ],
})

split_summary

,split,rows,positive_rate
0,train,108841,0.576465
1,validation,23323,0.577198
2,test,23324,0.578974


### Analysis

The split preserves temporal order:

- earliest 70% → training,
- next 15% → validation,
- latest 15% → test.

This is stricter than the random split used in the reference notebook and
better represents future production scoring.

## 2. One-hot encoding and numerical scaling

In [5]:
# -------------------
# Feature engineering
# -------------------

# `prepare_model_data` returns raw prediction-time features.
# The production model performs this transformation inside `fit_pipeline`.
# Here I run it explicitly only so the notebook can show each preprocessing step.

feature_engineer = ShipmentFeatureEngineer()

X_train_featured = feature_engineer.fit_transform(
    data.X_train
)

print("Raw train shape:", data.X_train.shape)
print("After feature engineering:", X_train_featured.shape)

engineered_columns = [
    column
    for column in X_train_featured.columns
    if column.startswith("order_")
    or column.endswith("_frequency")
]

engineered_columns

Raw train shape: (108841, 28)
After feature engineering: (108841, 39)


['order_item_discount',
 'order_item_discount_rate',
 'order_item_product_price',
 'order_item_profit_ratio',
 'order_item_quantity',
 'order_item_total_amount',
 'order_profit_per_order',
 'order_country',
 'order_region',
 'order_city',
 'order_state',
 'order_date',
 'order_year',
 'order_month',
 'order_quarter',
 'order_week',
 'order_day',
 'order_weekday',
 'order_hour',
 'order_is_weekend',
 'customer_city_frequency',
 'order_city_frequency',
 'order_state_frequency']

### Conclusion

The split intentionally returns raw prediction-time columns. After fitting
`ShipmentFeatureEngineer` on the training partition, the missing temporal and frequency
features now exist. This is the same transformer that `fit_pipeline()` persists with each
trained model, so validation and future API predictions reuse the training-time mappings.

In [6]:
# -------------------
# One-hot encoding and numerical preprocessing
# -------------------

preprocessor = build_preprocessor(
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    scale_numerical=True,
)

transformed = preprocessor.fit_transform(
    X_train_featured
)

print("Engineered train shape:", X_train_featured.shape)
print("Encoded train shape:", transformed.shape)

Engineered train shape: (108841, 39)
Encoded train shape: (108841, 427)


### Conclusion

The preprocessor now receives exactly the columns it expects. Numerical fields are
median-imputed and scaled for this demonstration; categorical fields are imputed and
one-hot encoded. The wider encoded matrix is expected because every retained category
becomes a model-readable indicator.

### Analysis

**Numerical pipeline**

median imputation → standard scaling

**Categorical pipeline**

most-frequent imputation → OneHotEncoder(handle_unknown="ignore")

One-hot encoding is appropriate for nominal categories such as payment type,
market, product category, and shipping mode because no artificial numeric order
should be imposed.

## 3. Encoded feature names

In [7]:
# -------------------
# Encoded feature names
# -------------------

feature_names = preprocessor.get_feature_names_out()

print("Total transformed features:", len(feature_names))
pd.DataFrame({"feature": feature_names}).head(100)

Total transformed features: 427


,feature
0,profit_per_order
1,sales_per_customer
2,latitude
3,longitude
4,order_item_discount
5,order_item_discount_rate
6,order_item_product_price
7,order_item_profit_ratio
8,order_item_quantity
9,sales


### Conclusion

The encoded feature list is the final numeric representation seen by the estimator.
Importantly, the model notebooks do **not** repeat these steps manually:
`fit_pipeline()` runs `ShipmentFeatureEngineer` → preprocessing → estimator as one saved
artifact.